# Exploration of Water Quality Station (WQS) Datasets

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the downloaded result records and selected station metadata.
PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "data/raw/wqp/results"
DOWNLOAD_DIR = RESULTS_DIR / "all_removals"

print("Results directory exists:", DOWNLOAD_DIR.exists())
print("Notebook project root:", PROJECT_ROOT)

Results directory exists: True
Notebook project root: /courses/DS5500.202710/teams/team4


In [2]:
manifest = pd.read_csv(DOWNLOAD_DIR / "download_manifest.csv")

display(manifest)

print("\nBatch status:")
display(manifest["status"].value_counts().to_frame("batches"))

if "rows" in manifest.columns:
    print("Total reported rows:", manifest["rows"].sum())

## Load and combine files:

In [3]:
# Find all batch CSVs, excluding the manifest.
batch_files = sorted(DOWNLOAD_DIR.glob("results_batch_*.csv"))

print(f"Found {len(batch_files)} batch files")

# Read each batch and retain its filename for troubleshooting.
frames = []

for file in batch_files:
    try:
        df = pd.read_csv(file, low_memory=False)
        df["source_file"] = file.name
        frames.append(df)
        print(f"{file.name}: {len(df):,} rows")
    except Exception as exc:
        print(f"Could not read {file.name}: {exc}")

if not frames:
    raise ValueError("No result batches could be read.")

results = pd.concat(frames, ignore_index=True)

print(f"\nCombined shape: {results.shape}")
print(f"Unique columns: {results.shape[1]}")

Found 0 batch files


ValueError: No result batches could be read.

In [ ]:
results.head()

In [ ]:
results.info()

In [ ]:
# Inspect the different measurement values:
print(results["CharacteristicName"].value_counts().head(100).to_string())

The results above show that there many different characteristics are measured across sites, and many of these measurements are not relevant to this study. Target columns will be filtered below:

In [ ]:
targets = [
    "Turbidity",
    "Temperature, water",
    "Specific conductance",
    "Oxygen",
    "Dissolved oxygen (DO)",
    "pH",
    "Nitrate",
    "Ammonia and ammonium",
    "Orthophosphate",
    "Phosphorus",
    "Total dissolved solids",
]

for target in targets:
    subset = results[results["CharacteristicName"] == target]

    print(
        f"{target:30s} "
        f"Results: {len(subset):5,} | "
        f"Stations: {subset['MonitoringLocationIdentifier'].nunique():3,} | "
        f"Dates: {subset['ActivityStartDate'].nunique():4,}"
    )

In [ ]:
# Examine sampling date range of pilot:
results["ActivityStartDate"] = pd.to_datetime(
    results["ActivityStartDate"],
    errors="coerce",
)

print("Earliest measurement:", results["ActivityStartDate"].min())
print("Latest measurement:", results["ActivityStartDate"].max())

## Now filter measurements within +/- 5 years of the dam removal

In [ ]:
# Compare downloaded station IDs with the current pilot selection.
selected_stations = pd.read_csv(
    RESULTS_DIR / "pilot_station_selection.csv",
    dtype={"MonitoringLocationIdentifier": "string"},
)

selected_ids = set(
    selected_stations["MonitoringLocationIdentifier"]
    .dropna()
    .str.strip()
)

downloaded_ids = set(
    results["MonitoringLocationIdentifier"]
    .dropna()
    .str.strip()
)

print("Selected stations:", len(selected_ids))
print("Downloaded stations:", len(downloaded_ids))
print("Selected but not downloaded:", len(selected_ids - downloaded_ids))
print("Downloaded but not selected:", len(downloaded_ids - selected_ids))

In [ ]:
# Download relevant data 
LINKS_DIR = PROJECT_ROOT / "data/processed"
DAMS_FILE = PROJECT_ROOT / "data/raw/dams/ARDamRemovalList_Figshare_Mar2026.csv"

link_files = sorted(LINKS_DIR.glob("dam_station_links_*.csv"))

links = pd.concat(
    [
        pd.read_csv(f, dtype={"MonitoringLocationIdentifier": "string"})
        for f in link_files
    ],
    ignore_index=True,
)

dams = pd.read_csv(DAMS_FILE)

print(f"Loaded {len(link_files)} link files")
print(f"Total link rows: {len(links):,}")

display(links.head())
display(dams.head())

print("Link columns:", links.columns.tolist())
print("Dam columns:", dams.columns.tolist())

In [ ]:
# Build dam-station relationships with removal dates.
dam_station_dates = links[
    [
        "dam_id",
        "dam_name",
        "station_id",
        "year_removed",
        "huc8",
        "relationship",
        "distance_to_dam_km",
        "network_distance_km",
    ]
].copy()

# Normalize station identifiers for joining.
dam_station_dates["MonitoringLocationIdentifier"] = (
    dam_station_dates["station_id"].astype("string").str.strip()
)

results["MonitoringLocationIdentifier"] = (
    results["MonitoringLocationIdentifier"].astype("string").str.strip()
)

# Compare station identifiers before the join.
print(
    "Unique downloaded station IDs:",
    results["MonitoringLocationIdentifier"].nunique(),
)

print(
    "Unique linked station IDs:",
    dam_station_dates["MonitoringLocationIdentifier"].nunique(),
)

print("\nSample downloaded IDs:")
print(results["MonitoringLocationIdentifier"].dropna().head(10).tolist())

print("\nSample linked IDs:")
print(
    dam_station_dates["MonitoringLocationIdentifier"]
    .dropna()
    .head(10)
    .tolist()
)

# Check overlap.
result_ids = set(results["MonitoringLocationIdentifier"].dropna())
linked_ids = set(
    dam_station_dates["MonitoringLocationIdentifier"].dropna()
)

print("\nMatching station IDs:", len(result_ids & linked_ids))
print("Downloaded IDs not linked:", len(result_ids - linked_ids))
print("Linked IDs not downloaded:", len(linked_ids - result_ids))

In [ ]:
# Summarize measurement coverage by calendar year.
year_counts = (
    results.assign(
        measurement_year=results["ActivityStartDate"].dt.year
    )
    .groupby("measurement_year")
    .size()
    .rename("measurement_count")
    .reset_index()
)

display(year_counts.tail(20))

In [ ]:
# Create +/-5 year subset
dam_station_dates = (
    links[
        ["dam_id", "station_id", "year_removed"]
    ]
    .rename(columns={"station_id": "MonitoringLocationIdentifier"})
    .drop_duplicates()
    .copy()
)

dam_station_dates["year_removed"] = pd.to_numeric(
    dam_station_dates["year_removed"],
    errors="coerce",
)

dam_station_dates = dam_station_dates.dropna(
    subset=["year_removed"]
).copy()

dam_station_dates["year_removed"] = (
    dam_station_dates["year_removed"].astype(int)
)

# Approximate removal date using January 1 of the recorded removal year.
dam_station_dates["removal_date"] = pd.to_datetime(
    dam_station_dates["year_removed"].astype(str) + "-01-01",
    errors="coerce",
)

# Parse measurement dates.
results["ActivityStartDate"] = pd.to_datetime(
    results["ActivityStartDate"],
    errors="coerce",
)

# Join measurements to their linked dam(s).
dam_results = results.merge(
    dam_station_dates,
    on="MonitoringLocationIdentifier",
    how="inner",
    validate="many_to_many",
)

# Filter to the inclusive +/-5 calendar-year window.
dam_results["window_start"] = (
    dam_results["removal_date"] - pd.DateOffset(years=5)
)
dam_results["window_end"] = (
    dam_results["removal_date"] + pd.DateOffset(years=5)
)

pilot_results = dam_results.loc[
    dam_results["ActivityStartDate"].between(
        dam_results["window_start"],
        dam_results["window_end"],
        inclusive="both",
    )
].copy()

print(f"All downloaded results: {len(results):,}")
print(f"Dam-linked measurement records: {len(dam_results):,}")
print(f"Records within +/-5 years: {len(pilot_results):,}")
print(
    "Unique stations in window:",
    pilot_results["MonitoringLocationIdentifier"].nunique(),
)
print("Unique dams represented:", pilot_results["dam_id"].nunique())

display(pilot_results.head(10))

In [ ]:
targets = [
    "Turbidity",
    "Temperature, water",
    "Specific conductance",
    "Oxygen",
    "Dissolved oxygen (DO)",
    "pH",
    "Nitrate",
    "Ammonia and ammonium",
    "Orthophosphate",
    "Phosphorus",
    "Total dissolved solids",
]

coverage = (
    pilot_results.groupby("CharacteristicName")
    .agg(
        results=("ActivityStartDate", "size"),
        stations=("MonitoringLocationIdentifier", "nunique"),
        dams=("dam_id", "nunique"),
        first_measurement=("ActivityStartDate", "min"),
        last_measurement=("ActivityStartDate", "max"),
    )
    .sort_values("results", ascending=False)
)

display(coverage)

for target in targets:
    subset = pilot_results[
        pilot_results["CharacteristicName"] == target
    ]

    print(
        f"{target:30s} "
        f"Results: {len(subset):6,} | "
        f"Stations: {subset['MonitoringLocationIdentifier'].nunique():4,} | "
        f"Dams: {subset['dam_id'].nunique():3,}"
    )

In [ ]:
# Inspect which dams have measurements in the analysis window.
display(
    pilot_results.groupby(["dam_id", "removal_date"])
    .agg(
        measurements=("ActivityStartDate", "size"),
        stations=("MonitoringLocationIdentifier", "nunique"),
        first_measurement=("ActivityStartDate", "min"),
        last_measurement=("ActivityStartDate", "max"),
    )
    .reset_index()
    .sort_values("measurements", ascending=False)
)

# Choose a dam ID from the table above.
selected_dam_id = pilot_results["dam_id"].dropna().iloc[0]

dam_measurements = (
    pilot_results[pilot_results["dam_id"] == selected_dam_id]
    .sort_values(
        ["MonitoringLocationIdentifier", "ActivityStartDate"]
    )
)

display(dam_measurements)

## Now check the consistency of measurement values

In [ ]:
pilot_results.columns

In [ ]:
t = pilot_results[pilot_results["CharacteristicName"].isin(targets)].copy()                                                                                                   
                                                                                                                                                                                
# 1. Filters + 2. detection/qualifier codes                                                                                                                                   
for col in ["ActivityTypeCode", "ActivityMediaName", "ActivityMediaSubdivisionName",                                                                                          
          "SampleAquifer", "ResultStatusIdentifier", "ResultValueTypeName",                                                                                                 
          "StatisticalBaseCode", "ResultDetectionConditionText",                                                                                                            
          "MeasureQualifierCode", "DetectionQuantitationLimitTypeName",                                                                                                     
          "HydrologicCondition", "HydrologicEvent", "ProviderName"]:                                                                                                        
  print(f"\n=== {col}")                                                                                                                                                     
  print(t[col].value_counts(dropna=False).head(25).to_string())      

In [ ]:
# 2. Non-numeric result values                                                                                                                                                
vals = t["ResultMeasureValue"].astype("string")                                                                                                                               
non_numeric = vals[pd.to_numeric(vals, errors="coerce").isna()]                                                                                                               
print("\n=== Non-numeric ResultMeasureValue")                                                                                                                                 
print(non_numeric.value_counts(dropna=False).head(30).to_string())

In [ ]:
# 3. Units per parameter + fraction                                                                                                                                           
display(                                                                                                                                                                      
  t.groupby(["CharacteristicName", "ResultSampleFractionText",                                                                                                              
             "ResultMeasure/MeasureUnitCode", "USGSPCode"], dropna=False)                                                                                                   
   .size().rename("rows").reset_index()                                                                                                                                     
)                                                                                                                                                                             
print(t.loc[t["CharacteristicName"] == "Specific conductance",                                                                                                                
          "ResultTemperatureBasisText"].value_counts(dropna=False))

In [ ]:
# 4. Candidate duplicates (same sample reported more than once)                                                                                                               
key = ["MonitoringLocationIdentifier", "ActivityStartDate", "ActivityStartTime/Time",                                                                                         
     "CharacteristicName", "ResultSampleFractionText", "ResultMeasureValue"]                                                                                                
dups = t[t.duplicated(key, keep=False)].sort_values(key)                                                                                                                      
print(f"\nDuplicate rows: {len(dups):,} of {len(t):,}")                                                                                                                       
display(dups[key + ["ProviderName", "OrganizationIdentifier", "ActivityIdentifier"]].head(20)) 

In [ ]:
pd.set_option("display.width", 250)

# Units table (cell 3), printed so it pastes cleanly
print(t.groupby(["CharacteristicName", "ResultSampleFractionText",
               "ResultMeasure/MeasureUnitCode", "USGSPCode"], dropna=False)
     .size().to_string())

In [ ]:
# What are the Calculated, Unknown and qualifier K/S2/H rows?
for mask, label in [(t["ResultValueTypeName"] == "Calculated", "Calculated"),
                  (t["ActivityTypeCode"] == "Unknown", "Unknown activity"),
                  (t["MeasureQualifierCode"].isin(["K", "S2", "H"]), "Qualifier K/S2/H")]:
  print(f"\n=== {label}")
  print(t[mask].groupby(["ProviderName", "OrganizationIdentifier", "CharacteristicName",
                         "MeasureQualifierCode"], dropna=False).size().to_string())

In [ ]:
# Duplicates remaining AFTER the draft filters, one row per measurement (ignore dam fan-out)
f = t[~t["ActivityTypeCode"].str.startswith("Quality Control", na=False)
    & (t["ActivityMediaName"] == "Water")
    & (t["ResultStatusIdentifier"] != "Rejected")]
f = f.drop_duplicates(["ActivityIdentifier", "CharacteristicName", "ResultSampleFractionText"])
key = ["MonitoringLocationIdentifier", "ActivityStartDate", "ActivityStartTime/Time",
     "CharacteristicName", "ResultSampleFractionText", "ResultMeasureValue"]
d = f[f.duplicated(key, keep=False)]
print(f"\nRows after filters: {len(f):,} | still duplicated: {len(d):,}")
print("Duplicate groups spanning both providers:",
    (d.groupby(key, dropna=False)["ProviderName"].nunique() > 1).sum())

In [ ]:
display(d.sort_values(key)[key + ["ProviderName", "ActivityTypeCode", "ActivityIdentifier"]].head(20))                          